####Setup

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"

display(
    spark.sql(
        f"SHOW TABLES IN {catalog}.{bronze_schema}"
    )
)

####Giottus spot balance

In [ ]:
balances_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.giottus_spot_balances"
)

print("Bronze balance rows:", balances_bronze_df.count())
balances_bronze_df.printSchema()


In [ ]:
giottus_balances_silver_df = (
    balances_bronze_df
    .select(
        col("source_file_name").alias("snapshot_id"),
        lit(None).cast("string").alias("account_id"),
        upper(col("asset")).alias("asset"),
        col("free").cast("decimal(38,18)").alias("free_balance"),
        col("locked").cast("decimal(38,18)").alias("locked_balance"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn(
        "total_balance",
        col("free_balance") + col("locked_balance")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

display(giottus_balances_silver_df)

In [ ]:
giottus_balances_final_df = (
    giottus_balances_silver_df
    .dropDuplicates(["snapshot_id", "asset", "account_id"])
)

(
    giottus_balances_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_account_balances")
)

print(
    "Saved Giottus balance rows:",
    spark.table(f"{catalog}.{silver_schema}.giottus_account_balances").count()
)


####Giottus trades

In [ ]:
spot_trades_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.giottus_spot_trades"
)

ebes_trades_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.giottus_ebes_trades"
)

print("spot trade rows:", spot_trades_bronze_df.count())
print("ebes trade rows:", ebes_trades_bronze_df.count())

In [ ]:
giottus_trades_raw_df = (
    spot_trades_bronze_df
    .withColumn("trade_source", lit("Spot"))
    .unionByName(
        ebes_trades_bronze_df
        .withColumn("trade_source", lit("EBES")),
        allowMissingColumns=True
    )
)

print("Combined trade rows:", giottus_trades_raw_df.count())

display(giottus_trades_raw_df)

In [ ]:
giottus_trade_parts_df = (
    giottus_trades_raw_df
    .withColumn("base_asset", split(col("symbol"), "/")[0])
    .withColumn("quote_asset", split(col("symbol"), "/")[1])
    .withColumn(
        "side",
        when(col("isBuyer") == True, "BUY").otherwise("SELL")
    )
    .withColumn("trade_time", timestamp_millis(col("time")))
)

display(
    giottus_trade_parts_df.select(
        "trade_source","symbol","side",
        "base_asset","quote_asset",
        "trade_time"
    )
)

giottus_trade_parts_df.printSchema()

In [ ]:
decimal_type = "decimal(38,18)"

giottus_trade_amounts_df = (
    giottus_trade_parts_df
    .withColumn(
        "asset_spent",
        when(col("side") == "BUY", col("quote_asset"))
        .otherwise(col("base_asset"))
    )
    .withColumn(
        "amount_spent",
        when(col("side") == "BUY", col("quoteQty"))
        .otherwise(col("qty"))
        .cast(decimal_type)
    )
    .withColumn(
        "asset_received",
        when(col("side") == "BUY", col("base_asset"))
        .otherwise(col("quote_asset"))
    )
    .withColumn(
        "amount_received",
        when(col("side") == "BUY", col("qty"))
        .otherwise(col("quoteQty"))
        .cast(decimal_type)
    )
)

display(
    giottus_trade_amounts_df.select(
        "trade_source", "side",
        "asset_spent", "amount_spent",
        "asset_received", "amount_received"
    )
)

In [ ]:
giottus_trades_silver_df = giottus_trade_amounts_df.select(
    col("id").cast("string").alias("transaction_id"),
    "trade_source",
    "symbol",
    "side",
    "asset_spent",
    "amount_spent",
    "asset_received",
    "amount_received",
    col("commission").cast(decimal_type).alias("fee_amount"),
    upper(col("commissionAsset")).alias("fee_asset"),
    "trade_time",
    "exchange",
    "_source_file",
    col("ingested_at").alias("bronze_ingested_at")
)

display(giottus_trades_silver_df)

In [ ]:
from pyspark.sql.functions import col, concat_ws, current_timestamp, to_date

giottus_trades_final_df = (
    giottus_trades_silver_df
    .filter(
        col("transaction_id").isNotNull()
        & col("asset_spent").isNotNull()
        & col("asset_received").isNotNull()
        & (col("amount_spent") > 0)
        & (col("amount_received") > 0)
    )
    .withColumn(
        "trade_key",
        concat_ws(
            "|", "exchange", "trade_source",
            "symbol", "transaction_id"
        )
    )
    .dropDuplicates(["trade_key"])
    .withColumn("trade_date", to_date(col("trade_time")))
    .withColumn("silver_processed_at", current_timestamp())
)

print("Clean Giottus trades:", giottus_trades_final_df.count())
display(giottus_trades_final_df)

In [ ]:
(
    giottus_trades_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_trades")
)

print(
    "Saved Giottus trades:",
    spark.table(f"{catalog}.{silver_schema}.giottus_trades").count()
)

####Giottus fiat withdrawals

In [ ]:
fiat_withdrawals_bronze_df = spark.table(
    f"{catalog}.bronze.giottus_fiat_withdrawals"
)

print("Fiat withdrawal rows:", fiat_withdrawals_bronze_df.count())
fiat_withdrawals_bronze_df.printSchema()

In [ ]:
from pyspark.sql.functions import (
    col, current_timestamp, lit, timestamp_millis, upper
)

fiat_withdrawals_silver_df = (
    fiat_withdrawals_bronze_df
    .select(
        col("referenceId").alias("withdrawal_id"),
        lit("INR").alias("asset"),
        col("amount").cast("decimal(38,18)").alias("amount"),
        col("fee").cast("decimal(38,18)").alias("fee_amount"),
        upper(col("status")).alias("status"),
        timestamp_millis(col("time")).alias("withdrawal_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

print("Clean fiat withdrawals:", fiat_withdrawals_silver_df.count())
fiat_withdrawals_silver_df.printSchema()

In [ ]:
(
    fiat_withdrawals_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_fiat_withdrawals")
)

print(
    "Saved fiat withdrawals:",
    spark.table(
        f"{catalog}.{silver_schema}.giottus_fiat_withdrawals"
    ).count()
)

####Giottus fiat deposits

In [ ]:
fiat_deposits_bronze_df = spark.table(
    f"{catalog}.bronze.giottus_fiat_deposits"
)

print("Fiat deposit rows:", fiat_deposits_bronze_df.count())
fiat_deposits_bronze_df.printSchema()

In [ ]:
fiat_deposits_silver_df = (
    fiat_deposits_bronze_df
    .select(
        col("referenceId").alias("deposit_id"),
        lit("INR").alias("asset"),
        col("amount").cast("decimal(38,18)").alias("amount"),
        col("fee").cast("decimal(38,18)").alias("fee_amount"),
        upper(col("status")).alias("status"),
        timestamp_millis(col("time")).alias("deposit_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

display(fiat_deposits_silver_df)

In [ ]:
fiat_deposits_final_df = (
    fiat_deposits_silver_df
    .dropDuplicates(["deposit_id"])
)

(
    fiat_deposits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_fiat_deposits")
)

print(
    "Saved fiat deposits:",
    spark.table(
        f"{catalog}.{silver_schema}.giottus_fiat_deposits"
    ).count()
)

####Giottus crypto withdrawals

In [ ]:
crypto_withdrawals_bronze_df = spark.table(
    f"{catalog}.bronze.giottus_crypto_withdrawals"
)

print(
    "Crypto withdrawal rows:",
    crypto_withdrawals_bronze_df.count()
)
crypto_withdrawals_bronze_df.printSchema()

In [ ]:
crypto_withdrawals_silver_df = (
    crypto_withdrawals_bronze_df
    .select(
        col("transactionHash").alias("withdrawal_id"),
        col("transactionHash").alias("transaction_hash"),
        upper(col("asset")).alias("asset"),
        col("cryptoAmount").cast("decimal(38,18)").alias("amount"),
        col("fee").cast("decimal(38,18)").alias("transaction_fee"),
        upper(col("network")).alias("network"),
        upper(col("status")).alias("status"),
        col("fromAddress").alias("from_address"),
        col("toAddress").alias("withdrawal_address"),
        timestamp_millis(col("time")).alias("withdrawal_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

display(crypto_withdrawals_silver_df)

In [ ]:
crypto_withdrawals_final_df = (
    crypto_withdrawals_silver_df
    .dropDuplicates(["withdrawal_id"])
)

(
    crypto_withdrawals_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_crypto_withdrawals")
)

print(
    "Saved crypto withdrawals:",
    spark.table(
        f"{catalog}.{silver_schema}.giottus_crypto_withdrawals"
    ).count()
)

####Giottus crypto deposits

In [ ]:
crypto_deposits_bronze_df = spark.table(
    f"{catalog}.bronze.giottus_crypto_deposits"
)

print("Crypto deposit rows:", crypto_deposits_bronze_df.count())
crypto_deposits_bronze_df.printSchema()

In [ ]:
crypto_deposits_silver_df = (
    crypto_deposits_bronze_df
    .select(
        col("transactionHash").alias("deposit_id"),
        col("transactionHash").alias("transaction_hash"),
        upper(col("asset")).alias("asset"),
        col("cryptoAmount").cast("decimal(38,18)").alias("amount"),
        col("fee").cast("decimal(38,18)").alias("fee_amount"),
        upper(col("network")).alias("network"),
        upper(col("status")).alias("status"),
        col("fromAddress").alias("source_address"),
        col("toAddress").alias("deposit_address"),
        timestamp_millis(col("time")).alias("deposit_time"),
        col("exchange"),
        col("_source_file"),
        col("ingested_at").alias("bronze_ingested_at")
    )
    .withColumn("silver_processed_at", current_timestamp())
)

print("Clean crypto deposits:", crypto_deposits_silver_df.count())
crypto_deposits_silver_df.printSchema()

In [ ]:
(
    crypto_deposits_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{catalog}.{silver_schema}.giottus_crypto_deposits")
)

print(
    "Saved crypto deposits:",
    spark.table(
        f"{catalog}.{silver_schema}.giottus_crypto_deposits"
    ).count()
)